# X-Phi Journals

## Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

from __future__ import annotations

# Standard library imports
from pathlib import Path
import re

# Third-party imports
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


#################
# CONFIGURATION #
#################

# Define periods
PERIODS = [
    "2001–2005",
    "2006–2010",
    "2011–2015",
    "2016–2020",
    "2021–2025"
]

# Define keywords indicating a source is a journal
JOURNAL_KEYWORDS = [
    "analysis",
    "bioethics",
    "cognition",
    "cognition and culture",
    "cognitive science",
    "consciousness",
    "cultura",
    "dialectica",
    "episteme",
    "erkenntnis",
    "ethics",
    "frontiers",
    "inquiry",
    "journal",
    "monist",
    "nous",
    "noûs",
    "personality and social psychology bulletin",
    "philosophia",
    "plos one",
    "psychological science",
    "quarterly",
    "review",
    "scientific reports",
    "studies",
    "synthese",
    "theoria",
    "topoi"
]

# Define keywords indicating a source is not a journal
NON_JOURNAL_KEYWORDS = [
    "advances in experimental philosophy",
    "anthology",
    "blackwell",
    "bloomsbury",
    "bloomsbury academic",
    "book",
    "brill",
    "cambridge university press",
    "companion",
    "compendium",
    "conference",
    "congress",
    "current controversies",
    "disputatio",
    "dissertatio",
    "dissertation",
    "doi.org",
    "edited by",
    "encyclopaedia",
    "encyclopedia",
    "festschrift",
    "handbook",
    "introduction",
    "isbn",
    "königshausen & neumann",
    "mit press",
    "moral psychology",
    "new waves",
    "oxford studies",
    "oxford university press",
    "palgrave",
    "palgrave macmillan",
    "palgrave mcmillan",
    "preprint",
    "press",
    "proceedings",
    "publishers",
    "publishing",
    "punctum",
    "routledge",
    "rowman & littlefield",
    "springer",
    "ssrn",
    "suhrkamp",
    "symposium",
    "thelem",
    "thesis",
    "transcript",
    "verlag",
    "volume",
    "vols",
    "workshop",
    "zenodo"
]

# Map journal names to standardized forms
JOURNAL_MAP = {
    "American Journal of Bioethics - Neuroscience": "American Journal of Bioethics Neuroscience",
    "Analyse and Kritik": "Analyse & Kritik",
    "Analyze and Kritik": "Analyse & Kritik",
    "Brain and Behavioral Sciences": "Behavioral and Brain Sciences",
    "Discipline filosofiche": "Discipline Filosofiche",
    "Erkenntnis (5": "Erkenntnis",
    "Ethics and Bioethics (in Central Europe": "Ethics and Bioethics (in Central Europe)",
    "Inquiry: An Interdisciplinary Journal of Philosophy:1-9": "Inquiry: An Interdisciplinary Journal of Philosophy",
    "International Journal for the Semiotics of Law - Revue Internationale de Sémiotique Juridique": "International Journal for the Semiotics of Law",
    "Journal for General Philosophy of Science / Zeitschrift für Allgemeine Wissenschaftstheorie": "Journal for General Philosophy of Science",
    "Journal of Aesthetics and Art Criticism": "Journal of Aesthetics and Art Criticism",
    "Journal of Experimental Psychology General": "Journal of Experimental Psychology: General",
    "Journal of Experimental Psychology. General": "Journal of Experimental Psychology: General",
    "Mind & Language": "Mind & Language",
    "Mind and Language": "Mind & Language",
    "Monist": "The Monist",
    "Nous": "Noûs",
    "Noûs": "Noûs",
    "O Gnition": "Cognition",
    "Philosophical Studies (6": "Philosophical Studies",
    "Philosophy & Phenomenological Research": "Philosophy and Phenomenological Research",
    "Philosophy and Phenomenological Research": "Philosophy and Phenomenological Research",
    "PLoS ONE": "PLOS ONE",
    "PLOS One": "PLOS ONE",
    "Plos One": "PLOS ONE",
    "Psychonomic Bulletin & Review": "Psychonomic Bulletin and Review",
    "Review of Philosophy & Psychology": "Review of Philosophy and Psychology",
    "Review of Philosophy and Psychology": "Review of Philosophy and Psychology",
    "Stanford Encyclopaedia of Philosophy": "Stanford Encyclopedia of Philosophy",
    "Stanford Encyclopedia of Philosophy": "Stanford Encyclopedia of Philosophy",
    "Studia Semiotyczne - English Supplement": "Studia Semiotyczne—English Supplement",
    "The Journal of Aesthetics and Art Criticism": "Journal of Aesthetics and Art Criticism",
    "The Monist": "The Monist",
    "Theoria : An International Journal for Theory, History and Fundations of Science": "Theoria",
    "Trends in Cognitive Science": "Trends in Cognitive Sciences",
}

## Define Helper Functions

In [ ]:
# Define function to replace multiple spaces with a single space
def compact_spaces(text: str) -> str:
    
    # Replace multiple spaces with a single space and strip leading or trailing spaces
    return re.sub(r"\s+", " ", str(text).strip())

# Define function to calculate likelihood of a string being a publication source
def candidate_source_score(candidate: str) -> int:
    
    # Strip leading and trailing spaces
    c = candidate.strip()
    
    # Convert to lowercase
    cl = c.lower()
    
    # Initialize score
    score = 0
    
    # Add points if string contains "In"
    if c.startswith("In "): score += 6
    
    # Add points if string matches volume-issue pattern
    if re.search(r"\b\d{1,4}\s*(?:\([^)]*\))?\s*:", c): score += 5
    
    # Add points if string matches volume-issue pattern without colon
    if re.search(r"\b\d{1,4}\s*\([^)]*\)", c): score += 4
    
    # Add points if string ends with standalone number
    if re.search(r"\b\d{1,4}\s*$", c): score += 2
    
    # Add points if string contains "dissertation"
    if "dissertation" in cl: score += 6
    
    # Add points if string contains publisher keywords
    if any(k in cl for k in [
        "blackwell",
        "bloomsbury",
        "bloomsbury academic",
        "brill",
        "cambridge university press",
        "königshausen & neumann",
        "mit press",
        "oup",
        "oxford university press",
        "palgrave",
        "palgrave macmillan",
        "palgrave mcmillan",
        "press",
        "publishers",
        "publishing",
        "punctum",
        "routledge",
        "rowman & littlefield",
        "springer",
        "suhrkamp",
        "thelem",
        "transcript",
        "verlag"
    ]):
        score += 4
    
    # Add points if string contains journal keywords
    if any(k in cl for k in JOURNAL_KEYWORDS): score += 3
    
    # Add point if string starts with capital letter
    if re.match(r"^[A-ZÁÉÍÓÚÄÖÜÀÈÌÒÙÂÊÎÔÛÆØÅÇÑŠŽ]", c): score += 1
    
    # Subtract points if string starts with question word
    if re.match(r"^(Why|What|How|When|Where|Which|Who|Whom|Whose)\b", c): score -= 3
    
    # Return score
    return score

# Define function to split string into title and source
def split_title_source(rest: str) -> tuple[str, str, str, int]:
    
    # Apply compact_spaces and remove leading periods
    rest = compact_spaces(re.sub(r"^\.\s*", "", rest))
    
    # Identify punctuation boundaries
    boundaries = list(re.finditer(r"(?<=[.!?])\s+", rest))
    
    # Return entire string as title if no boundaries found
    if not boundaries:
        return rest.strip(" ."), "", "No punctuation boundary", 0
    
    # Initialize best split
    best = None
    
    # Iterate through punctuation boundaries
    for m in boundaries:
        
        # Extract potential title
        title = rest[:m.start() + 1].strip()
        
        # Extract potential source
        source = rest[m.end():].strip()
        
        # Apply candidate_source_score
        score = candidate_source_score(source)
        
        # Check if score meets minimum threshold
        if score >= 5:
            
            # Update best split if score is higher or if it is tied but occurs later in string
            if best is None or score > best[0] or (score == best[0] and m.start() > best[1]):
                best = (score, m.start(), title, source)
    
    # Return best split if found
    if best:
        return best[2].strip(" ."), best[3].strip(), "", best[0]
    
    # Fall back to last punctuation mark if no best split is found
    m = boundaries[-1]
    
    # Return fallback split
    return rest[:m.start() + 1].strip(" ."), rest[m.end():].strip(), "Low-confidence split", 0

# Define function to clean up book titles
def extract_book_chapter_source(source: str) -> str:
    
    # Apply compact_spaces
    s = compact_spaces(source)
    
    # Return string if it does not start with "In"
    if not s.startswith("In "): return s
    
    # Remove "In"
    s = s[3:].strip()
    
    # Remove page numbers
    s = re.sub(r"\s+pp\.\s+.*$", "", s).strip()
    
    # Split by comma and take second part to remove editor names
    if "," in s: s = s.split(",", 1)[1].strip()
    
    # Split remaining string at first period
    parts = re.split(r"\.\s+", s)
    
    # Return clean book title
    return parts[0].strip(" .,:;") if parts else s.strip(" .,:;")

# Define function to remove volume, issue, and page numbers
def strip_volume_issue_pages(source: str) -> str:
    
    # Apply compact_spaces and remove trailing punctuation mark
    s = compact_spaces(source).strip(" .,:;-")
    
    # Remove volume-issue-page patterns
    s = re.sub(r"\s+\d{1,4}\s*\([A-Za-z0-9\s\–\-\—]*[\)\-]?.*$", "", s)
    s = re.sub(r"\s+\d{1,4}\s*[-–—]+.*$", "", s)
    s = re.sub(r"\s*\(?\d+\)?\s*:\s*\d*[-–—]*\d*.*$", "", s)
    s = re.sub(r":\s*\d+[-–—]\d+.*$", "", s)
    s = re.sub(r"\s+\d{1,4}\s*\([^)]*\)\s*:\s*[\w–—\-]+.*$", "", s)
    s = re.sub(r"\s+\d{1,4}\s*:\s*[\w–—\-]+.*$", "", s)
    s = re.sub(r"\s+\d{1,4}\s*\([^)]*\)\s*$", "", s)
    s = re.sub(r"\s+\d{1,4}\s*$", "", s)
    s = re.sub(r"\s+\([^)]*$", "", s)
    
    # Return cleaned string and strip remaining trailing punctuation or empty parentheses
    return s.strip(" .,:;-()")

# Define function to classify source
def classify_source_type(standardized: str, original: str) -> str:
    
    # Apply compact_spaces and convert standardized string to lowercase
    s = compact_spaces(standardized).lower()
    
    # Apply compact_spaces and convert original string to lowercase
    o = compact_spaces(original).lower()
    
    # Create combined string
    combined = f"{s} {o}"
    
    # Flag as unknown if string is empty
    if not s:
        return "Unknown"
    
    # Reject strings too short to be a journal name
    if len(s) <= 3:
        return "Other"
    
    # Reject strings containing URLs, DOIs, repositories, page ranges, ISBNs, or prices
    if re.search(r"(https?://|doi\.org|zenodo|ssrn|\d{2,}–\d{2,}|isbn|€|£|\$)", s):
        return "Other"
    
    # Reject strings matching a city-publisher pattern
    if re.search(r"^[A-Z][a-zäöüß]+(?:, \w+)?: ", standardized):
        return "Other"
    
    # Reject title-journal mashups indicated by a period
    if ". " in standardized:
        return "Other"
    
    # Reject long strings starting with "A", "The", or "Some"
    if standardized.startswith(("a ", "the ", "some ")) and len(standardized) > 40:
        return "Other"
    
    # Reject long strings containing "of the" or "on the"
    if " of the " in s or " on the " in s:
        if len(s) > 40:
            return "Other"
    
    # Reject if matched with NON_JOURNAL_KEYWORDS
    if any(k in combined for k in NON_JOURNAL_KEYWORDS):
        return "Other"
    
    # Approve if matched with JOURNAL_KEYWORDS
    if any(k in combined for k in JOURNAL_KEYWORDS):
        return "Likely journal"
    
    # Approve if original string contains volume-issue pattern
    if re.search(r"\b\d{1,4}\s*(?:\([^)]*\))?\s*:", original):
        return "Likely journal"
    
    # Flag for review if no conditions are met
    return "Needs review"

# Define function to parse, clean, map, and classify source
def standardize_source(source: str | None, score: int = 0) -> tuple[str, str]:
    
    # Return "Unknown" if source is empty or None
    if source is None or not str(source).strip():
        return "", "Unknown"
    
    # Apply compact_spaces and strip leading or trailing periods
    original = compact_spaces(source).strip(" .")
    
    # Assign to working variable
    s = original
    
    # Check if source starts with "In "
    if s.startswith("In "):
        
        # Apply extract_book_chapter_source
        s = extract_book_chapter_source(s)
    
    # Otherwise treat as likely journal
    else:
        
        # Clean up parsing artifacts
        for marker in [". Cognitive Science", ". Philosophical Psychology", ". Synthese", ". Cognition", ". Analysis", ". Mind & Language", ". Mind and Language"]:
            if marker in s:
                s = marker[2:]
                break
        
        # Apply strip_volume_issue_pages
        s = strip_volume_issue_pages(s)
    
    # Apply compact_spaces and remove trailing punctuation
    s = compact_spaces(s).strip(" .,:;")
    
    # Map standardized name to dictionary version
    s = JOURNAL_MAP.get(s, s)
    
    # Apply classify_source_type to standardized and original string
    stype = classify_source_type(s, original)
    
    # Return name and classification
    return s, stype

## Analyze Data

In [ ]:
# Define function to parse entries into authors, year, title, and source
def parse_reference(ref: str) -> dict:
    
    # Locate publication year
    match = re.search(r"[\(\[]\b((?:19|20)\d{2})\b[\)\]]", ref)
    
    # Flag entry as invalid if no publication year is found
    if not match:
        return {
            "Authors": "Unknown",
            "Year":    0,
            "Source":  "",
            "Period":  None,
            "Score":   0
        }
    
    # Extract publication year
    year = match.group(1)
    
    # Split entry into authors and rest
    parts = re.split(re.escape(match.group(0)), ref, maxsplit = 1)
    
    # Strip trailing or leading spaces and punctuation
    authors = parts[0].strip(" .")
    rest = parts[1].strip(" .") if len(parts) > 1 else ""
    
    # Apply split_title_source to rest
    title, source, _, score = split_title_source(rest)
    
    # Assign time period
    year_int = int(year)
    period = next((p for p in PERIODS if int(p[:4]) <= year_int <= int(p[-4:])), None)
    
    # Return dictionary
    return {
        "Authors": authors,
        "Year":    year_int,
        "Title":   title,
        "Source":  source,
        "Period":  period,
        "Score":   score
    }

# Define function to read, parse, standardize, and aggregate entries
def run_analysis():
    
    # Check if file exists
    p = Path("x_phi_journals_philpapers.txt")
    if not p.exists(): return print("File is missing"), None
    
    # Read file line by line
    lines = [line.strip() for line in p.read_text(encoding = "utf-8").splitlines() if line.strip()]
    
    # Apply parse_reference
    df = pd.DataFrame([parse_reference(line) for line in lines])
    
    # Drop rows where year parsing failed
    df = df[df["Year"] > 0].copy()
    
    # Apply standardize_source
    std = df.apply(lambda r: standardize_source(r["Source"], r["Score"]), axis = 1)
    
    # Extract resulting tuple elements into DataFrame columns
    df["Standardized source"] = std.apply(lambda x: x[0])
    df["Source type"] = std.apply(lambda x: x[1])
    
    # Isolate entries that are likely from journals
    data = df[df["Source type"] == "Likely journal"].copy()
    
    # Drop entries where journal name is empty
    data = data[data["Standardized source"].str.strip().ne("")]
    
    # Stop if filtering did not leave any data
    if data.empty:
        print("No journals identified")
        return None, df
    
    # Create frequency matrix (rows = journals, columns = time periods, values = publication count)
    rank = pd.crosstab(data["Standardized source"], data["Period"])
    
    # Ensure all time periods exist as columns
    for p_col in PERIODS:
        if p_col not in rank.columns: rank[p_col] = 0
    
    # Reorder columns chronologically
    rank = rank[PERIODS]
    
    # Calculate total publications per journal
    rank["Total"] = rank.sum(axis = 1)
    
    # Sort DataFrame
    rank = rank.reset_index().sort_values(["Total"] + PERIODS[::-1], ascending = [False] * 6).reset_index(drop = True)
    
    # Insert ranking number as first column
    rank.insert(0, "Rank", range(1, len(rank) + 1))

    # Return rank and processed DataFrame
    return rank, df

# Run function to read, parse, standardize, and aggregate entries
rank_table, full_df = run_analysis()

# Output top t25 journals
if rank_table is not None:
    display(rank_table.head(25))

# Output summary statistics and manual review queue
if full_df is not None:
    
    # Count sources
    total_entries = len(full_df)
    
    journal_count = len(full_df[full_df["Source type"] == "Likely journal"])
    other_count   = len(full_df[full_df["Source type"] == "Other"])
    unknown_count = len(full_df[full_df["Source type"] == "Unknown"])
    review_count  = len(full_df[full_df["Source type"] == "Needs review"])
    unique_count  = full_df[full_df["Source type"] == "Likely journal"]["Standardized source"].nunique()
    
    # Calculate percentages
    def calc_pct(count, total):
        return (count / total * 100) if total > 0 else 0.0
    
    journal_pct = calc_pct(journal_count, total_entries)
    other_pct   = calc_pct(other_count, total_entries)
    unknown_pct = calc_pct(unknown_count, total_entries)
    review_pct  = calc_pct(review_count, total_entries)
    
    # Print summary
    print("Summary:")
    print(f"Total entries:         {total_entries}")
    print(f"Likely journal:        {journal_count} ({journal_pct:.3f}%)")
    print(f"Likely non-journal:    {other_count} ({other_pct:.3f}%)")
    print(f"Unknown source:        {unknown_count} ({unknown_pct:.3f}%)")
    print(f"Needs manual review:   {review_count} ({review_pct:.3f}%)")
    print(f"Distinct journals:     {unique_count}")
    
    # Print manual review queue
    print("Manual review queue:")
    needs_review = full_df[full_df["Source type"].isin(["Needs review"])]
    display(needs_review)

## Extract Unique Sources

In [ ]:
# Filter DataFrame to only include likely journals
journal_df = full_df[full_df["Source type"] == "Likely journal"]

# Extract unique sources
unique_sources = journal_df[journal_df["Standardized source"].str.strip() != ""]["Standardized source"].unique()

# Sort alphabetically
unique_sources.sort()

# Define output file name
output_file_name = "x_phi_journals_unique_sources.txt"

# Write unique sources to text file
with open(output_file_name, "w", encoding = "utf-8") as file:
    for source in unique_sources:
        file.write(f"{source}\n")

# Print confirmation
print(f"Saved {len(unique_sources)} unique sources to '{output_file_name}'.")

## Generate Graphs

In [ ]:
# Define visualization function
def visualize_trends(df_rank):
    
    # Check if DataFrame is missing or empty
    if df_rank is None or df_rank.empty:
        return
    
    # Isolate top 25 and prepare data
    top25 = df_rank.head(25).set_index("Standardized source")[PERIODS]
    
    # Generate heatmap
    plt.figure(figsize = (16, 10))
    sns.heatmap(top25, annot = True, cmap = "YlGnBu", fmt = "g", cbar_kws = {'label': 'Publications'})
    plt.title("Top 25 Journals Over Time", pad = 20)
    plt.ylabel("Number of Publications")
    plt.ylabel("")
    
    # Export heatmap
    plt.tight_layout()
    plt.savefig("x_phi_journals_heatmap.pdf", bbox_inches = "tight")
    plt.show()
    
    # Generate line chart
    plt.figure(figsize = (16, 10))
    df_line = top25.T
    max_pubs = df_line.max()
    highlight_mask = max_pubs > 10
    highlight_count = highlight_mask.sum()
    highlight_colors = sns.color_palette("husl", highlight_count)
    final_colors = []
    color_index = 0
    
    for journal in df_line.columns:
        if highlight_mask[journal]:
            final_colors.append(highlight_colors[color_index])
            color_index += 1
        else:
            final_colors.append("#d3d3d3")
    
    df_line.plot(kind = "line", marker = "o", color = final_colors, ax = plt.gca())
    plt.title("Top 25 Journals Over Time", pad = 20)
    plt.ylabel("Number of Publications")
    plt.grid(True, linestyle = "--", alpha = 0.7)
    plt.legend(title = "Journals", bbox_to_anchor = (1.05, 1), loc = 'upper left')
    
    # Export line chart
    plt.tight_layout()
    plt.savefig("x_phi_journals_line_chart.pdf", bbox_inches = "tight")
    plt.show()

# Run visualization function
visualize_trends(rank_table)